# 1. Data: sources, cohorts, endpoint and split

This notebook prepares everything that comes before the model. We download each data source and record its version, build the two patient cohorts, define the outcome we predict, fix the list of candidate genes and set aside a test set. Every later notebook starts from the files written here.

Patient-level data stays in the `data/` folder, which git ignores. The outputs below are counts, summaries and plots.

## Settings

Every choice of this notebook is named in the second cell below, with its reason beside it. The first cell only loads the libraries and locates the folders.

In [ ]:
import hashlib
import json
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor
from datetime import date
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split
from sksurv.nonparametric import kaplan_meier_estimator


def git(*arguments):
    """The output of a git command run in this repository."""
    completed = subprocess.run(["git", *arguments], capture_output=True, text=True, check=True)
    return completed.stdout.strip()


REPO = Path(git("rev-parse", "--show-toplevel"))
DATA = REPO / "data"                    # downloads and patient-level tables; ignored by git
RESULTS = REPO / "results" / "01_data"  # aggregate tables only; tracked by git
RESULTS.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", None)  # tables hold long text such as checksums; show it
HTTP = requests.Session()               # one session, so that connections to a server are reused

In [ ]:
GDC_API = "https://api.gdc.cancer.gov"  # NCI Genomic Data Commons
GDC_PAGE_SIZE = 10000                   # larger than any file list we ask for: one page holds all
TCGA_PROJECT = "TCGA-BRCA"
CDR_FILE_ID = "1b5f413e-a8d1-4d10-92eb-7c4ae739ed81"  # the TCGA-CDR workbook, PanCanAtlas page
SUPPLEMENT_NAME = "nationwidechildrens.org_clinical_patient_brca.txt"  # patient table, supplement
DATAHUB = "cBioPortal/datahub/dca75cb3f32b82d54a6f78bf0a6323e5b975aca1/public/brca_metabric"
# The datahub commit above was pinned on 2026-09-30.
HTTP_TIMEOUT = (10, 600)                # seconds to wait for a connection, then for data; the short
                                        # first value moves on when one address does not answer
DOWNLOAD_THREADS = 8                    # parallel downloads; the expression source has 1,000 files
CHUNK_BYTES = 1 << 20                   # files are read and written in pieces of 1 MiB
PRIMARY_TUMOUR_CODE = "01"              # TCGA sample-type code of a primary solid tumour
DAYS_PER_YEAR = 365.25                  # average length of a year, leap years included
DAYS_PER_MONTH = DAYS_PER_YEAR / 12     # METABRIC reports follow-up in months
CLASS_CUTS_DAYS = [365, 1095, 1825]     # 1, 3 and 5 years: class borders of the earlier pipeline
TEST_FRACTION = 0.2                     # share of TCGA patients locked away as the test set
SPLIT_SEED = 0                          # fixed so that the split is the same in every run

## Downloading with a record

Four helpers serve every source. For each source we first write a manifest: the list of its files, the checksum the source publishes for each one, the version and the date. A checksum is a short fingerprint of a file's content. Files are downloaded only when they are missing, so a second run downloads nothing, and every file is compared with its checksum in every run.

In [ ]:
def file_hash(path, algorithm):
    """The checksum of a file, read in pieces."""
    digest = hashlib.new(algorithm)
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(CHUNK_BYTES), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download(url, path):
    """Fetch url into path unless the file is already there; True when a download happened."""
    if path.exists():
        return False
    partial = path.with_name(path.name + ".part")  # a half-written file must never look complete
    with HTTP.get(url, stream=True, timeout=HTTP_TIMEOUT) as response:
        response.raise_for_status()
        with open(partial, "wb") as handle:
            for chunk in response.iter_content(CHUNK_BYTES):
                handle.write(chunk)
    partial.rename(path)
    return True

In [ ]:
def write_manifest(folder, source_version, files):
    """Record the file list of a source with its version and the date of retrieval."""
    folder.mkdir(parents=True, exist_ok=True)
    manifest = {"version": source_version, "retrieved_on": date.today().isoformat(), "files": files}
    (folder / "manifest.json").write_text(json.dumps(manifest, indent=1))


def matches_checksum(folder, entry):
    """Whether a downloaded file has the checksum its source published."""
    return file_hash(folder / entry["name"], entry["checksum_type"]) == entry["checksum"]


def fetch_and_verify(folder, files):
    """Download the missing files, then count the files that match their recorded checksum."""
    with ThreadPoolExecutor(DOWNLOAD_THREADS) as pool:
        fetched = pool.map(lambda entry: download(entry["url"], folder / entry["name"]), files)
        downloaded = sum(fetched)
        verified = sum(pool.map(lambda entry: matches_checksum(folder, entry), files))
    return {"downloaded in this run": downloaded, "checksum correct": verified, "files": len(files)}

## Source 1: TCGA-BRCA gene expression

The expression data come from the NCI Genomic Data Commons (GDC). We ask the GDC for every open-access STAR gene-count file of the project (STAR is the program that aligns the sequencing reads). Each file holds, for one tissue sample, the read count and the TPM value of every gene. TPM (transcripts per million) is a count scaled for gene length and sequencing depth, so that samples are comparable. The answer is saved as the manifest, with the data release the GDC reports. The output counts the listed files by sample type.

In [ ]:
def gdc_list_files(filters, fields):
    """Ask the GDC files index for the files that match every filter."""
    conditions = []
    for field, value in filters.items():
        condition = {"op": "=", "content": {"field": field, "value": value}}
        conditions.append(condition)
    body = {"filters": {"op": "and", "content": conditions}, "fields": ",".join(fields),
            "size": GDC_PAGE_SIZE}
    answer = HTTP.post(f"{GDC_API}/files", json=body, timeout=HTTP_TIMEOUT).json()["data"]
    complete = answer["pagination"]["total"] == len(answer["hits"])
    assert complete, "the file list must arrive complete, not as a first page"
    return answer["hits"]


def gdc_release():
    """The data release the GDC currently serves."""
    return HTTP.get(f"{GDC_API}/status", timeout=HTTP_TIMEOUT).json()["data_release"]


def star_file_entry(hit):
    """One manifest entry of an expression file: name, address, checksum, patient, sample."""
    return {"name": hit["file_name"], "url": f"{GDC_API}/data/{hit['file_id']}",
            "checksum_type": "md5", "checksum": hit["md5sum"],
            "patient": hit["cases"][0]["submitter_id"],
            "sample_type": hit["cases"][0]["samples"][0]["sample_type"],
            "aliquot": hit["associated_entities"][0]["entity_submitter_id"]}

In [ ]:
star_dir = DATA / "raw" / "gdc_star_counts"
if not (star_dir / "manifest.json").exists():
    filters = {"cases.project.project_id": TCGA_PROJECT,
               "data_type": "Gene Expression Quantification",
               "analysis.workflow_type": "STAR - Counts", "access": "open"}
    fields = ["file_id", "file_name", "md5sum", "cases.submitter_id", "cases.samples.sample_type",
              "associated_entities.entity_submitter_id"]
    files = []
    for hit in gdc_list_files(filters, fields):
        files.append(star_file_entry(hit))
    write_manifest(star_dir, gdc_release(), sorted(files, key=lambda entry: entry["aliquot"]))
star_listing = pd.DataFrame(json.loads((star_dir / "manifest.json").read_text())["files"])
star_listing["sample_type"].value_counts().rename("files").to_frame()

## One primary tumour sample per patient

We keep primary tumour samples only, and exactly one per patient. Normal tissue and metastases differ biologically from the primary tumour, and a patient with two samples would count twice. The GDC records how each sample was preserved: FFPE (formalin-fixed, paraffin-embedded) tissue yields degraded RNA, frozen tissue (OCT) does not. We fetch the preservation method of the primary tumour files once and save it beside the manifest, which does not change. The table counts the files of patients with several files by vial letter (the sample copy in the barcode) and preservation method.

In [ ]:
def gdc_preservation(file_ids):
    """The preservation method the GDC records for the sample of each given file."""
    chosen = {"op": "in", "content": {"field": "file_id", "value": sorted(file_ids)}}
    body = {"filters": chosen, "fields": "file_id,cases.samples.preservation_method",
            "size": GDC_PAGE_SIZE}
    answer = HTTP.post(f"{GDC_API}/files", json=body, timeout=HTTP_TIMEOUT).json()["data"]
    assert len(answer["hits"]) == len(file_ids), "the preservation list must arrive complete"
    methods = {}
    for hit in answer["hits"]:
        methods[hit["file_id"]] = hit["cases"][0]["samples"][0]["preservation_method"]
    return methods


primary = star_listing[star_listing["sample_type"] == "Primary Tumor"].copy()
primary["file_id"] = primary["url"].str.rsplit("/", n=1).str[1]
preservation_path = star_dir / "preservation.json"
if not preservation_path.exists():
    record = {"retrieved_on": date.today().isoformat(), "release": gdc_release(),
              "preservation_method": gdc_preservation(primary["file_id"])}
    preservation_path.write_text(json.dumps(record, indent=1))
preservation = json.loads(preservation_path.read_text())["preservation_method"]
primary["preservation"] = primary["file_id"].map(preservation)
primary["vial"] = primary["aliquot"].str[15]      # the letter after the sample-type code
primary["plate"] = primary["aliquot"].str[20:24]  # sequencing plates are numbered in order
several = primary[primary.duplicated("patient", keep=False)]
duplicate_table = pd.crosstab(several["vial"], several["preservation"])
duplicate_table.index.name = "vial letter"
duplicate_table.columns.name = "preservation method"
duplicate_table.to_csv(RESULTS / "duplicate_samples.csv")
duplicate_table

When a patient has several primary tumour files, the FFPE copies are set aside and the original vial is kept; where two aliquots of that vial remain, the one from the earlier plate is kept. An aliquot is a portion of the sample that was sequenced separately. This fixed rule reads the sample records, not the expression values. The check confirms that it selects the same files as the rule used before, which kept the file whose barcode sorts first, so the cohort below is unchanged.

In [ ]:
def choose_file(files):
    """Set the FFPE copies of one patient aside, then keep the aliquot from the earlier plate."""
    if len(files) == 1:
        return files
    kept = files[files["preservation"] != "FFPE"]
    assert kept["plate"].is_unique, "two remaining aliquots come from the same plate"
    return kept.sort_values("plate").head(1)


chosen_files = []
for patient, files in primary.groupby("patient"):
    chosen_files.append(choose_file(files))
tcga_samples = pd.concat(chosen_files).set_index("patient").sort_index()
first_barcode = primary.sort_values("aliquot").groupby("patient").head(1)
same_files = set(tcga_samples["name"]) == set(first_barcode["name"])
assert same_files, "the stated rule selects other files than the barcode-first rule"
patients_with_several = int(several["patient"].nunique())
display(Markdown(f"{len(primary)} primary tumour files belong to {len(tcga_samples)} patients. "
                 f"{patients_with_several} patients have more than one file; for each of them one "
                 "file is kept by the rule above."))

A patient whose only primary tumour file is FFPE keeps that file, because the rule sets FFPE copies aside only where another file exists. The sentence counts these files. They stay in this data version; whether to exclude them is an open item for the supervisors.

In [ ]:
ffpe_selected = int((tcga_samples["preservation"] == "FFPE").sum())
display(Markdown(f"{ffpe_selected} of the {len(tcga_samples)} selected files are FFPE; each is the "
                 "only primary tumour file of its patient. They stay in this data version; "
                 "excluding them is an open item."))

In [ ]:
# The aliquot barcode carries the sample type in characters 14 and 15, independently of the GDC
# field used above.
sample_codes = tcga_samples["aliquot"].str[13:15]
assert (sample_codes == PRIMARY_TUMOUR_CODE).all(), "a selected sample is not a primary tumour"
assert tcga_samples.index.is_unique, "a patient has more than one selected sample"

Only the selected files are downloaded, several at a time, and each is compared with the MD5 checksum the GDC lists for it. The output says how many files were downloaded in this run and how many passed the check. The same cell stores the description of the source for the table further down.

In [ ]:
SOURCES = {}
SOURCES["TCGA-BRCA gene expression"] = {
    "folder": star_dir,
    "what it is": "STAR gene counts and TPM per Ensembl gene (GENCODE v36), one file per tissue "
                  "sample",
    "where from": f"{GDC_API}/files with project {TCGA_PROJECT}, Gene Expression Quantification, "
                  f"STAR - Counts, open access; each file from {GDC_API}/data/[file id]",
    "terms of use": "Open access, no login. GDC open data fall under the NIH Genomic Data Sharing "
                    "policy: do not attempt to identify participants, and acknowledge the data set "
                    "and the repository. TCGA asks for an acknowledgement of the TCGA Research "
                    "Network (https://www.cancer.gov/tcga).",
    "citation": "Heath et al., Nat Genet 2021, doi:10.1038/s41588-021-00791-5 (GDC); "
                "The Cancer Genome Atlas Network, Nature 2012, doi:10.1038/nature11412 (TCGA-BRCA)",
    "source checksum": "MD5 listed in the GDC files index",
    "check": fetch_and_verify(star_dir, tcga_samples.to_dict("records"))}
SOURCES["TCGA-BRCA gene expression"]["check"]

## Source 2: TCGA outcomes

The outcomes come from the TCGA Pan-Cancer Clinical Data Resource (CDR), a curated table of follow-up for all TCGA patients published with Liu et al. (2018). The GDC distributes it as one workbook on its PanCanAtlas publication page and sends the MD5 checksum of the file in the response header, which we record as the source checksum.

In [ ]:
cdr_dir = DATA / "raw" / "tcga_cdr"
cdr_url = f"{GDC_API}/data/{CDR_FILE_ID}"
if not (cdr_dir / "manifest.json").exists():
    with HTTP.get(cdr_url, stream=True, timeout=HTTP_TIMEOUT) as response:  # headers only here
        cdr_md5 = response.headers["Content-MD5"]
    cdr_files = [{"name": "TCGA-CDR-SupplementalTableS1.xlsx", "url": cdr_url,
                  "checksum_type": "md5", "checksum": cdr_md5}]
    write_manifest(cdr_dir, "Supplemental Table S1 of Liu et al. 2018, GDC file " + CDR_FILE_ID,
                   cdr_files)
cdr_manifest = json.loads((cdr_dir / "manifest.json").read_text())
SOURCES["TCGA outcomes (CDR)"] = {
    "folder": cdr_dir,
    "what it is": "Curated clinical table of all TCGA cancer types with four outcome endpoints "
                  "(OS, DSS, DFI, PFI)",
    "where from": cdr_url + ", linked from "
                  "https://gdc.cancer.gov/about-data/publications/pancanatlas",
    "terms of use": "Open access at the GDC under the same policy as the expression files. "
                    "The article is open access under CC BY-NC-ND 4.0.",
    "citation": "Liu et al., Cell 2018, doi:10.1016/j.cell.2018.02.052",
    "source checksum": "MD5 sent by the GDC in the Content-MD5 response header",
    "check": fetch_and_verify(cdr_dir, cdr_manifest["files"])}
SOURCES["TCGA outcomes (CDR)"]["check"]

## Source 3: TCGA receptor status

The CDR does not carry the receptor status of the tumour (ER, PR and HER2), which are standard clinical variables in breast cancer. They are in the clinical supplement of TCGA-BRCA at the GDC, a tab-separated table with one row per patient. We look the file up by name, so its checksum comes from the GDC index.

In [ ]:
supplement_dir = DATA / "raw" / "gdc_clinical_supplement"
if not (supplement_dir / "manifest.json").exists():
    filters = {"cases.project.project_id": TCGA_PROJECT, "file_name": SUPPLEMENT_NAME,
               "access": "open"}
    hit, = gdc_list_files(filters, ["file_id", "file_name", "md5sum"])  # fails unless one file
    supplement_files = [{"name": hit["file_name"], "url": f"{GDC_API}/data/{hit['file_id']}",
                         "checksum_type": "md5", "checksum": hit["md5sum"]}]
    write_manifest(supplement_dir, gdc_release(), supplement_files)
supplement_manifest = json.loads((supplement_dir / "manifest.json").read_text())
SOURCES["TCGA-BRCA clinical supplement"] = {
    "folder": supplement_dir,
    "what it is": "Patient table of the clinical supplement (BCR Biotab), used for ER, PR and "
                  "HER2 status",
    "where from": f"{GDC_API}/files with project {TCGA_PROJECT} and file name {SUPPLEMENT_NAME}; "
                  f"the file from {GDC_API}/data/[file id]",
    "terms of use": "Open access at the GDC under the same policy as the expression files.",
    "citation": "The Cancer Genome Atlas Network, Nature 2012, doi:10.1038/nature11412; "
                "Heath et al., Nat Genet 2021, doi:10.1038/s41588-021-00791-5 (GDC)",
    "source checksum": "MD5 listed in the GDC files index",
    "check": fetch_and_verify(supplement_dir, supplement_manifest["files"])}
SOURCES["TCGA-BRCA clinical supplement"]["check"]

## Source 4: METABRIC

METABRIC is the independent cohort: breast tumours collected in the United Kingdom and Canada and measured on a microarray instead of by sequencing. We take it from the cBioPortal datahub on GitHub, pinned to one commit so that the files cannot change under us. The datahub stores large files through Git LFS, whose pointer file records the SHA-256 checksum of the real file; we use it as the source checksum.

In [ ]:
metabric_dir = DATA / "raw" / "metabric"
metabric_names = ["data_clinical_patient.txt", "data_clinical_sample.txt",
                  "data_mrna_illumina_microarray.txt"]
if not (metabric_dir / "manifest.json").exists():
    files = []
    for name in metabric_names:
        pointer_url = f"https://raw.githubusercontent.com/{DATAHUB}/{name}"
        pointer = HTTP.get(pointer_url, timeout=HTTP_TIMEOUT).text
        checksum = pointer.split("oid sha256:")[1].split()[0]
        entry = {"name": name, "url": f"https://media.githubusercontent.com/media/{DATAHUB}/{name}",
                 "checksum_type": "sha256", "checksum": checksum}
        files.append(entry)
    write_manifest(metabric_dir, "cBioPortal datahub commit " + DATAHUB.split("/")[2], files)
metabric_manifest = json.loads((metabric_dir / "manifest.json").read_text())
datahub_address = DATAHUB.replace(DATAHUB.split("/")[2], " [commit] ")
SOURCES["METABRIC"] = {
    "folder": metabric_dir,
    "what it is": "Clinical patient table, clinical sample table and Illumina HT-12 v3 expression "
                  "matrix (log2 intensity) of the study brca_metabric",
    "where from": "https://media.githubusercontent.com/media/ " + datahub_address + "/ [file name]",
    "terms of use": "Open Database License (ODbL) 1.0, stated in the LICENSE file of the study "
                    "folder: free to share and adapt with attribution; derived data sets stay open "
                    "under the same licence.",
    "citation": "Curtis et al., Nature 2012, doi:10.1038/nature10983; Pereira et al., Nat Commun "
                "2016, doi:10.1038/ncomms11479 (METABRIC); Cerami et al., Cancer Discov 2012, "
                "doi:10.1158/2159-8290.CD-12-0095 (cBioPortal)",
    "source checksum": "SHA-256 recorded in the Git LFS pointer file",
    "check": fetch_and_verify(metabric_dir, metabric_manifest["files"])}
SOURCES["METABRIC"]["check"]

In [ ]:
# A file that differs from the checksum its source publishes is not the file the source describes.
for source in SOURCES.values():
    check = source["check"]
    assert check["checksum correct"] == check["files"], "a downloaded file fails its checksum"

## Data sources

The table gathers what a reader needs to obtain the same files: what each source is, where it comes from, its version, the day we retrieved it, its terms of use as the source states them, and the paper to cite. The last rows come from the local record: where the checksum of each file comes from, how many files passed it, and the SHA-256 of the manifest, which identifies the exact file list.

In [ ]:
def describe_source(name, source):
    """One row of the data sources table."""
    manifest = json.loads((source["folder"] / "manifest.json").read_text())
    return {"source": name, "what it is": source["what it is"], "where from": source["where from"],
            "version": manifest["version"], "retrieved on": manifest["retrieved_on"],
            "terms of use": source["terms of use"], "citation": source["citation"],
            "source checksum": source["source checksum"],
            "files checked": source["check"]["checksum correct"],
            "manifest sha256": file_hash(source["folder"] / "manifest.json", "sha256")}


def break_long_hex(column):
    """A line break after 32 characters of a checksum or commit, so that four columns fit a page."""
    return column.str.replace("([0-9a-f]{32})(?=[0-9a-f])", r"\1<br>", regex=True)


source_rows = []
for name, source in SOURCES.items():
    source_rows.append(describe_source(name, source))
sources_table = pd.DataFrame(source_rows).set_index("source")
sources_table.to_csv(RESULTS / "sources.csv")
# A Markdown table wraps long sentences; a data frame would cut or stretch them.
shown = sources_table.astype(str).apply(break_long_hex)
lines = ["| | " + " | ".join(shown.index) + " |", "|---" * (len(shown) + 1) + "|"]
for field in shown.columns:
    lines.append(f"| {field} | " + " | ".join(shown[field]) + " |")
display(Markdown("\n".join(lines)))

## TCGA-BRCA cohort

The cohort starts from the TCGA-BRCA patients in the CDR and applies five rules in order. Patients carrying the redaction flag of the CDR are dropped: TCGA marked these cases as problematic, for example for an uncertain sample identity. Men are dropped because the external cohort is all female. The patient must have a primary tumour RNA-seq file, a recorded overall survival (OS) status and time, and a time above zero, since a time of zero carries no survival information. No missing outcome is filled in. The table shows the patients and deaths left after each rule.

In [ ]:
def apply_rules(table, start_label, rules):
    """Apply the rules in order. Return the rows that remain and the counts after each rule."""
    start = {"rule": start_label, "patients": len(table), "deaths": int(table["os_event"].sum())}
    counts = [start]
    for label, keep in rules:
        table = table[keep(table)]
        count = {"rule": label, "patients": len(table), "deaths": int(table["os_event"].sum())}
        counts.append(count)
    return table.sort_index(), pd.DataFrame(counts).set_index("rule")


cdr = pd.read_excel(cdr_dir / "TCGA-CDR-SupplementalTableS1.xlsx", sheet_name="TCGA-CDR",
                    index_col="bcr_patient_barcode")
cdr_names = {"OS": "os_event", "OS.time": "os_days"}
cdr = cdr[cdr["type"] == "BRCA"].rename(columns=cdr_names)
tcga_rules = [
    ("without the redaction flag", lambda t: t["Redaction"].isna()),
    ("women", lambda t: t["gender"] == "FEMALE"),
    ("with a primary tumour RNA-seq file", lambda t: t.index.isin(tcga_samples.index)),
    ("with OS status and OS time recorded", lambda t: t["os_event"].notna() & t["os_days"].notna()),
    ("with OS time above 0", lambda t: t["os_days"] > 0)]
tcga, tcga_counts = apply_rules(cdr, "TCGA-BRCA patients in the CDR", tcga_rules)
tcga_counts.to_csv(RESULTS / "cohort_tcga.csv")
tcga_counts

In [ ]:
# The two exclusions must hold in the cohort that every later step uses.
assert tcga["Redaction"].isna().all(), "a patient with the redaction flag is in the cohort"
assert (tcga["gender"] == "FEMALE").all(), "a male patient is in the cohort"
without_record = int((~tcga_samples.index.isin(cdr.index)).sum())
display(Markdown("RNA-seq patients without a CDR record, and so without an outcome: "
                 f"{without_record}."))

## METABRIC cohort

The corresponding rules apply to METABRIC: the patient has a column in the expression matrix, the OS status and time are recorded, and the time is above zero. A recorded tumour stage is not required. Follow-up is given in months and converted to days. METABRIC outcomes are only counted in this notebook; no model sees them before the external evaluation.

In [ ]:
metabric_patients = pd.read_csv(metabric_dir / "data_clinical_patient.txt", sep="\t", comment="#",
                                index_col="PATIENT_ID")
metabric_samples = pd.read_csv(metabric_dir / "data_clinical_sample.txt", sep="\t", comment="#",
                               index_col="PATIENT_ID")
metabric_expression = pd.read_csv(metabric_dir / "data_mrna_illumina_microarray.txt", sep="\t")
status_codes = {"0:LIVING": 0, "1:DECEASED": 1}
metabric_patients["os_event"] = metabric_patients["OS_STATUS"].map(status_codes)
metabric_patients["os_days"] = metabric_patients["OS_MONTHS"] * DAYS_PER_MONTH
metabric_rules = [
    ("with a column in the expression matrix", lambda t: t.index.isin(metabric_expression.columns)),
    ("with OS status and OS time recorded", lambda t: t["os_event"].notna() & t["os_days"].notna()),
    ("with OS time above 0", lambda t: t["os_days"] > 0)]
metabric_start = "METABRIC patients in the clinical table"
metabric, metabric_counts = apply_rules(metabric_patients, metabric_start, metabric_rules)
metabric_counts.to_csv(RESULTS / "cohort_metabric.csv")
metabric_counts

In [ ]:
# The expression columns are named by sample; joining them to patients is only valid if each patient
# has one sample of the same name.
same_name = metabric_samples["SAMPLE_ID"] == metabric_samples.index
assert same_name.all(), "a METABRIC sample is not named like its patient"
assert metabric_samples.index.is_unique, "a METABRIC sample appears twice"
assert metabric_patients.index.is_unique, "a METABRIC patient appears twice"
# Men are excluded from the TCGA cohort because the external cohort has none; that must be true.
assert (metabric_patients["SEX"] == "Female").all(), "the METABRIC cohort is not all female"

## Clinical covariates

Five clinical variables are kept for both cohorts: age at diagnosis, tumour stage, and the status of three receptors (ER, PR and HER2) that guide breast cancer treatment. Stage is reduced to its main level. A receptor status is positive or negative; a test that was not done, or that gave an equivocal or indeterminate result, becomes the level unknown. For HER2 in TCGA a second test (FISH) exists, and it decides where the first test (IHC) gives no clear answer. Nobody is dropped for a missing covariate, and a missing age stays missing here.

In [ ]:
def receptor_status(status):
    """Keep positive and negative; every other value, missing ones included, becomes 'unknown'."""
    spelling = {"positve": "positive"}  # the METABRIC table spells it this way
    status = status.str.lower().replace(spelling)
    return status.where(status.isin(["positive", "negative"]), "unknown")


# Lines 2 and 3 of the file repeat the header in two other vocabularies and are skipped.
supplement = pd.read_csv(supplement_dir / SUPPLEMENT_NAME, sep="\t", skiprows=[1, 2],
                         index_col="bcr_patient_barcode")
supplement = supplement.reindex(tcga.index)  # a patient absent from the supplement gets 'unknown'
her2_by_ihc = receptor_status(supplement["her2_status_by_ihc"])
her2_by_fish = receptor_status(supplement["her2_fish_status"])
stage_pattern = r"^Stage (IV|III|II|I)"
tcga_stage = tcga["ajcc_pathologic_tumor_stage"].str.extract(stage_pattern, expand=False)
tcga_columns = {
    "os_days": tcga["os_days"], "os_event": tcga["os_event"].astype(int),
    "age": tcga["age_at_initial_pathologic_diagnosis"],
    "stage": tcga_stage.fillna("unknown"),
    "er": receptor_status(supplement["er_status_by_ihc"]),
    "pr": receptor_status(supplement["pr_status_by_ihc"]),
    "her2": her2_by_ihc.where(her2_by_ihc != "unknown", her2_by_fish)}  # FISH decides unclear cases
tcga_clinical = pd.DataFrame(tcga_columns)
metabric_sample = metabric_samples.loc[metabric.index]
stage_names = {0: "0", 1: "I", 2: "II", 3: "III", 4: "IV"}
metabric_columns = {
    "os_days": metabric["os_days"], "os_event": metabric["os_event"].astype(int),
    "age": metabric["AGE_AT_DIAGNOSIS"],
    "stage": metabric_sample["TUMOR_STAGE"].map(stage_names).fillna("unknown"),
    "er": receptor_status(metabric["ER_IHC"]),
    "pr": receptor_status(metabric_sample["PR_STATUS"]),
    "her2": receptor_status(metabric_sample["HER2_STATUS"])}
metabric_clinical = pd.DataFrame(metabric_columns)

Immunohistochemistry (IHC) stains the HER2 protein on the cell surface and is read as a score from 0 to 3+, where 2+ is equivocal. Fluorescence in situ hybridization (FISH) counts the copies of the HER2 gene in the nucleus. The HER2 testing guideline of the American Society of Clinical Oncology and the College of American Pathologists (Wolff et al., J Clin Oncol 2013, doi:10.1200/JCO.2013.50.9984; focused update 2018, doi:10.1200/JCO.2018.77.8738) calls for the other test when a result is equivocal; our rule keeps that order. The table crosses the two results for the cohort patients.

In [ ]:
def test_levels(status):
    """The recorded level of a HER2 test; a test not done or not recorded is one level."""
    levels = ["Positive", "Negative", "Equivocal", "Indeterminate"]
    return status.where(status.isin(levels), "not done or missing").str.lower()


ihc_levels = test_levels(supplement["her2_status_by_ihc"])
fish_levels = test_levels(supplement["her2_fish_status"])
level_order = ["positive", "negative", "equivocal", "indeterminate", "not done or missing"]
ihc_by_fish = pd.crosstab(ihc_levels, fish_levels).reindex(index=level_order, columns=level_order)
ihc_by_fish = ihc_by_fish.fillna(0).astype(int)
ihc_by_fish.index.name = "IHC result"
ihc_by_fish.columns.name = "FISH result"
ihc_by_fish.to_csv(RESULTS / "her2_ihc_by_fish.csv")
display(ihc_by_fish)
clear = ["positive", "negative"]
both_clear = int(ihc_by_fish.loc[clear, clear].to_numpy().sum())
agree = int(ihc_by_fish.loc["positive", "positive"] + ihc_by_fish.loc["negative", "negative"])
ihc_positive_only = int(ihc_by_fish.loc["positive", "negative"])
fish_positive_only = int(ihc_by_fish.loc["negative", "positive"])
display(Markdown(f"{both_clear} patients have a clear result on both tests: {agree} agree, "
                 f"{ihc_positive_only} are positive by IHC and negative by FISH, and "
                 f"{fish_positive_only} are negative by IHC and positive by FISH. For all of "
                 "them the rule follows IHC."))

The next table shows what the HER2 rule changes in TCGA: the level counts by IHC alone and after FISH has decided the unclear cases.

In [ ]:
her2_columns = {"IHC only": her2_by_ihc.value_counts(),
                "IHC, then FISH": tcga_clinical["her2"].value_counts()}
her2_rule_table = pd.DataFrame(her2_columns)
her2_rule_table.index.name = "HER2 level"
her2_rule_table.to_csv(RESULTS / "her2_rule.csv")
her2_rule_table

The table counts the patients at each level. In TCGA ER and PR are measured by immunohistochemistry (IHC) and HER2 by the rule above. In METABRIC only ER is stated to be by IHC; the file does not say how its PR and HER2 status were measured. A level that only one cohort uses will need a decision before the external evaluation.

In [ ]:
covariate_names = ["stage", "er", "pr", "her2"]
tcga_levels = {}
metabric_levels = {}
for name in covariate_names:
    tcga_levels[name] = tcga_clinical[name].value_counts()
    metabric_levels[name] = metabric_clinical[name].value_counts()
level_columns = {"TCGA-BRCA": pd.concat(tcga_levels), "METABRIC": pd.concat(metabric_levels)}
covariate_table = pd.DataFrame(level_columns).fillna(0).astype(int)
covariate_table.index.names = ["covariate", "level"]
covariate_table.to_csv(RESULTS / "covariates.csv")
display(covariate_table)
tcga_age_missing = int(tcga_clinical["age"].isna().sum())
metabric_age_missing = int(metabric_clinical["age"].isna().sum())
display(Markdown(f"Median age at diagnosis: {tcga_clinical['age'].median():.0f} years in TCGA-BRCA "
                 f"and {metabric_clinical['age'].median():.0f} in METABRIC. Age is missing for "
                 f"{tcga_age_missing} and {metabric_age_missing} patients."))

## Unknown values and follow-up

An unknown value can predict death for the wrong reason: if it marks patients from an earlier period, who were followed longer and had more time to die, a model can use it without any biology behind it. The table gives, for every level of the four categorical covariates in TCGA, the patients, the deaths and the median follow-up. The follow-up is estimated by the reverse Kaplan-Meier method, which treats the end of observation as the event and a death as an interruption. Compare each unknown level with the other levels of its covariate.

In [ ]:
def median_follow_up(clinical):
    """Median follow-up in years by reverse Kaplan-Meier; missing when under half reach the end."""
    still_observed_flag = clinical["os_event"] == 0
    years_observed = clinical["os_days"] / DAYS_PER_YEAR
    years, still_observed = kaplan_meier_estimator(still_observed_flag, years_observed)
    if (still_observed <= 0.5).any():
        return years[still_observed <= 0.5][0]
    return np.nan


def follow_up_row(clinical):
    """Patients, deaths, share of deaths and median follow-up of one group of patients."""
    return {"patients": len(clinical), "deaths": int(clinical["os_event"].sum()),
            "deaths (%)": round(100 * clinical["os_event"].mean(), 1),
            "median follow-up (years)": round(median_follow_up(clinical), 1)}


follow_up_rows = {}
for name in covariate_names:
    for level, group in tcga_clinical.groupby(name):
        follow_up_rows[(name, level)] = follow_up_row(group)
follow_up_table = pd.DataFrame.from_dict(follow_up_rows, orient="index")
follow_up_table.index.names = ["covariate", "level"]
follow_up_table.to_csv(RESULTS / "unknown_values_and_follow_up.csv")
display(follow_up_table)
unknown = follow_up_table.xs("unknown", level="level")["median follow-up (years)"]
display(Markdown(f"Median follow-up is {median_follow_up(tcga_clinical):.1f} years in the whole "
                 f"cohort. Among patients with an unknown value it is {unknown['stage']} years for "
                 f"stage, {unknown['er']} for ER, {unknown['pr']} for PR and {unknown['her2']} for "
                 "HER2. A missing follow-up in the table means that most patients of that level "
                 "died while under observation."))

## The endpoint: overall survival as time and event

The outcome is overall survival: the time from diagnosis to death from any cause. For a patient who was alive at the last contact we only know that survival is longer than the follow-up; such a record is called censored. We keep the outcome as two numbers per patient, the time in days and an event flag (1 for death, 0 for censored), and never turn it into a class. The table summarises both cohorts.

In [ ]:
def outcome_summary(clinical):
    """Patients, deaths, share censored and the observed times of one cohort."""
    return {"patients": len(clinical), "deaths": int(clinical["os_event"].sum()),
            "censored (%)": round(100 * (1 - clinical["os_event"].mean()), 1),
            "median observed time (years)": round(clinical["os_days"].median() / DAYS_PER_YEAR, 1),
            "longest observed time (years)": round(clinical["os_days"].max() / DAYS_PER_YEAR, 1)}


outcome_rows = [outcome_summary(tcga_clinical), outcome_summary(metabric_clinical)]
outcome_table = pd.DataFrame(outcome_rows, index=["TCGA-BRCA", "METABRIC"])
outcome_table.to_csv(RESULTS / "outcome_summary.csv")
outcome_table

The Kaplan-Meier curve estimates, from censored data, the share of patients still alive at each time. The figure answers one question: how survival and the length of follow-up differ between the two cohorts. The shaded band is the 95% confidence interval; it widens where few patients remain under observation.

In [ ]:
figure, axis = plt.subplots()
for name, clinical in [("TCGA-BRCA", tcga_clinical), ("METABRIC", metabric_clinical)]:
    died = clinical["os_event"].astype(bool)
    years, alive, band = kaplan_meier_estimator(died, clinical["os_days"] / DAYS_PER_YEAR,
                                                conf_type="log-log")
    label = f"{name}: {len(clinical)} patients, {clinical['os_event'].sum()} deaths"
    axis.step(years, alive, where="post", label=label)
    axis.fill_between(years, band[0], band[1], step="post", alpha=0.2)  # light shading keeps both
axis.set_xlabel("Time since diagnosis (years)")
axis.set_ylabel("Probability of being alive")
axis.set_ylim(0, 1)
axis.legend()
plt.show()

The CDR paper (Liu et al., 2018) advises caution with OS and DSS in breast cancer, because the follow-up of TCGA is short relative to the course of the disease, and recommends PFI and DFI. OS remains the working endpoint of this notebook. The table counts the events of every available endpoint among the cohort patients, so the cost of that choice is visible. DSS counts deaths from the disease, DFI a return of the disease after a disease-free period, and PFI any progression, return, or death from the disease.

In [ ]:
vital_known = metabric["VITAL_STATUS"].notna()
died_of_disease = (metabric["VITAL_STATUS"] == "Died of Disease").where(vital_known)
relapse_codes = {"0:Not Recurred": 0, "1:Recurred": 1}
endpoint_flags = {
    ("TCGA-BRCA", "OS: death from any cause"): tcga["os_event"],
    ("TCGA-BRCA", "DSS: death from the disease"): tcga["DSS"],
    ("TCGA-BRCA", "DFI: return after a disease-free period"): tcga["DFI"],
    ("TCGA-BRCA", "PFI: progression, return or death from the disease"): tcga["PFI"],
    ("METABRIC", "OS: death from any cause"): metabric["os_event"],
    ("METABRIC", "death from the disease"): died_of_disease,
    ("METABRIC", "relapse"): metabric["RFS_STATUS"].map(relapse_codes)}
endpoint_columns = {}
for key, flag in endpoint_flags.items():
    endpoint_columns[key] = {"patients with a value": int(flag.notna().sum()),
                             "events": int(flag.sum())}
endpoint_table = pd.DataFrame(endpoint_columns).T
endpoint_table.index.names = ["cohort", "endpoint"]
endpoint_table.to_csv(RESULTS / "endpoints.csv")
endpoint_table

Under overall survival a death from any cause is an event. Under disease-specific survival a death from another cause ends the follow-up without an event, like a last contact alive; in neither case is a patient removed. The table counts the deaths of each cohort by cause. The CDR approximates the cause from the vital status and the tumour status at death, so a TCGA death without a disease-specific value has no cause; METABRIC records the cause directly.

In [ ]:
tcga_deaths = tcga.loc[tcga["os_event"] == 1, "DSS"]
metabric_deaths = metabric.loc[metabric["os_event"] == 1, "VITAL_STATUS"]
cause_counts = {}
cause_counts[("TCGA-BRCA", "from the disease")] = int((tcga_deaths == 1).sum())
other_cause = "from another cause, as the CDR approximates it"
cause_counts[("TCGA-BRCA", other_cause)] = int((tcga_deaths == 0).sum())
cause_counts[("TCGA-BRCA", "no value")] = int(tcga_deaths.isna().sum())
disease_deaths = int((metabric_deaths == "Died of Disease").sum())
other_deaths = int((metabric_deaths == "Died of Other Causes").sum())
cause_counts[("METABRIC", "from the disease")] = disease_deaths
cause_counts[("METABRIC", "from other causes")] = other_deaths
cause_counts[("METABRIC", "cause not recorded")] = int(metabric_deaths.isna().sum())
deaths_by_cause = pd.Series(cause_counts, name="deaths").to_frame()
deaths_by_cause.index.names = ["cohort", "cause of death"]
deaths_by_cause.to_csv(RESULTS / "deaths_by_cause.csv")
deaths_by_cause

## Survival classes of the earlier pipeline

The earlier pipeline replaced the outcome by four classes of OS time, cut at 1, 3 and 5 years, and used the class number as the target of gene selection and as the stratification key. The table bins our TCGA cohort the same way. A censored patient in a class did not die in that interval: the follow-up ended there. A class made mostly of censored patients describes how long patients were followed, not how long they lived. For this reason the rebuild uses no classes and keeps time and event.

In [ ]:
class_labels = ["up to 1 year", "1 to 3 years", "3 to 5 years", "over 5 years"]
class_bins = [0, *CLASS_CUTS_DAYS, np.inf]
time_class = pd.cut(tcga_clinical["os_days"], bins=class_bins, labels=class_labels)
by_class = tcga_clinical.groupby(time_class, observed=False)["os_event"]
class_table = by_class.agg(patients="size", deaths="sum")
class_table["censored"] = class_table["patients"] - class_table["deaths"]
class_table["censored (%)"] = (100 * class_table["censored"] / class_table["patients"]).round(1)
class_table.index.name = "OS time"
class_table.to_csv(RESULTS / "survival_classes.csv")
display(class_table)
shortest = class_table.iloc[0]
display(Markdown(f"In the class {class_table.index[0]}, {shortest['censored']:.0f} of "
                 f"{shortest['patients']:.0f} patients ({shortest['censored (%)']}%) are censored. "
                 "They did not die within the year; their follow-up ended there."))

## Expression matrices

We read the TPM column of every cohort patient's file into one matrix, patients by genes, and take log2(TPM + 1), the usual transform that compresses the range of values; it applies to each value alone. The values are per gene: the GDC computes each gene's TPM from its read count and length, so the unit's name does not mean transcript level. No step here looks across patients: filtering and scaling belong to the training folds of the next notebook. Genes are keyed by Ensembl ID. The assert confirms that every file lists the same genes in the same order.

In [ ]:
def read_star_file(name, columns):
    """The gene rows of one expression file; its first four rows are mapping statistics."""
    table = pd.read_csv(star_dir / name, sep="\t", comment="#", usecols=columns)
    return table[~table["gene_id"].str.startswith("N_")].reset_index(drop=True)


tcga_genes = read_star_file(tcga_samples["name"].iloc[0], ["gene_id", "gene_name", "gene_type"])
tpm = {}
same_genes = []
for patient, name in tcga_samples.loc[tcga.index, "name"].items():
    table = read_star_file(name, ["gene_id", "tpm_unstranded"])
    same_genes.append(table["gene_id"].equals(tcga_genes["gene_id"]))
    tpm[patient] = table["tpm_unstranded"].to_numpy()
tcga_expression = np.log2(pd.DataFrame(tpm, index=tcga_genes["gene_id"]).T + 1)
display(Markdown(f"TCGA-BRCA expression matrix: {tcga_expression.shape[0]} patients by "
                 f"{tcga_expression.shape[1]} genes. METABRIC expression matrix: {len(metabric)} "
                 f"cohort patients by {len(metabric_expression)} rows."))

In [ ]:
# Columns of different files can only be stacked into one matrix if they describe the same genes in
# the same order.
assert all(same_genes), "an expression file lists different genes"
assert tcga_expression.notna().all().all(), "a TCGA expression value is missing"

## Candidate gene pool

A model trained on TCGA is later applied to METABRIC, so a gene is only usable if both platforms measure it. TCGA names genes by Ensembl ID and METABRIC by gene symbol, two naming systems for genes. We keep the genes whose symbol occurs exactly once in the TCGA gene list and exactly once in the METABRIC matrix, which makes the match one to one. The pool is decided from the two gene lists alone: no expression value and no outcome is used. The table shows what each matching step removes.

In [ ]:
metabric_symbol_counts = metabric_expression["Hugo_Symbol"].value_counts()
once_in_tcga = ~tcga_genes["gene_name"].duplicated(keep=False)
in_metabric = tcga_genes["gene_name"].isin(metabric_symbol_counts.index)
single_symbols = metabric_symbol_counts[metabric_symbol_counts == 1].index
once_in_metabric = tcga_genes["gene_name"].isin(single_symbols)
steps = {"genes in the TCGA files": pd.Series(True, index=tcga_genes.index),
         "symbol occurs once in TCGA": once_in_tcga,
         "symbol is present in METABRIC": once_in_tcga & in_metabric,
         "symbol occurs once in METABRIC": once_in_tcga & once_in_metabric}
genes_left = {}
for label, keep in steps.items():
    genes_left[label] = int(keep.sum())
pool_columns = {"genes left": genes_left}
pool_steps = pd.DataFrame(pool_columns)
pool_steps["removed by this step"] = (-pool_steps["genes left"].diff()).fillna(0).astype(int)
pool_names = {"gene_name": "symbol", "gene_type": "biotype"}
gene_pool = tcga_genes[once_in_tcga & once_in_metabric].rename(columns=pool_names)
gene_pool.to_csv(RESULTS / "gene_pool.csv", index=False)
pool_steps.to_csv(RESULTS / "gene_pool_steps.csv")
display(pool_steps)
protein_coding = (gene_pool["biotype"] == "protein_coding").mean()
display(Markdown(f"The candidate pool holds {len(gene_pool)} genes; {protein_coding:.1%} of them "
                 "are protein coding."))

Most genes without a METABRIC partner are not protein coding: the microarray was designed around known protein-coding genes, while the sequencing files list every annotated gene, including non-coding RNAs and pseudogenes. The table splits each matching step into protein-coding and other genes; the sentence gives the counts behind the split.

In [ ]:
is_protein_coding = tcga_genes["gene_type"] == "protein_coding"
biotype_rows = {}
for label, keep in steps.items():
    protein_coding_left = int((keep & is_protein_coding).sum())
    biotype_rows[label] = {"protein coding": protein_coding_left,
                           "other": int(keep.sum()) - protein_coding_left}
biotype_steps = pd.DataFrame(biotype_rows).T
biotype_steps.index.name = "step"
biotype_steps.to_csv(RESULTS / "gene_pool_biotypes.csv")
display(biotype_steps)
without_partner = once_in_tcga & ~in_metabric
share_other = 1 - (without_partner & is_protein_coding).sum() / without_partner.sum()
unmatched_metabric = int((~metabric_symbol_counts.index.isin(tcga_genes["gene_name"])).sum())
display(Markdown(f"The TCGA files hold {int(is_protein_coding.sum())} protein-coding genes and the "
                 f"pool keeps {biotype_steps['protein coding'].iloc[-1]} of them. Of the "
                 f"{int(without_partner.sum())} genes whose symbol has no METABRIC partner, "
                 f"{share_other:.1%} are not protein coding. {unmatched_metabric} METABRIC symbols "
                 "find no TCGA partner."))

## One table per cohort

Each cohort becomes one table with a row per patient: the outcome, the five covariates and the expression of the pool genes. The METABRIC columns are renamed from symbol to Ensembl ID, so both tables share their column names. METABRIC values are log2 microarray intensities and stay as they are. The two platforms are on different scales, which the scaling inside the training folds has to handle later.

In [ ]:
tcga_table = tcga_clinical.join(tcga_expression[gene_pool["gene_id"]])
by_symbol = metabric_expression.set_index("Hugo_Symbol")
metabric_pool = by_symbol.loc[gene_pool["symbol"], metabric.index].T
metabric_pool.columns = gene_pool["gene_id"].to_numpy()
metabric_table = metabric_clinical.join(metabric_pool)
missing_values = int(metabric_pool.isna().sum().sum())
display(Markdown(f"TCGA-BRCA table: {tcga_table.shape[0]} patients by {tcga_table.shape[1]} "
                 "columns. "
                 f"METABRIC table: {metabric_table.shape[0]} patients by {metabric_table.shape[1]} "
                 f"columns, with {missing_values} missing expression values left as they are."))

In [ ]:
# No step between the source tables and the cohort tables may fill in or change an outcome.
outcome = ["os_days", "os_event"]
tcga_same = tcga_table[outcome].eq(cdr.loc[tcga_table.index, outcome]).all().all()
assert tcga_same, "a TCGA outcome differs from the CDR"
assert tcga_table[outcome].notna().all().all(), "a TCGA outcome is missing"
metabric_source = metabric_patients.loc[metabric_table.index, outcome]
assert metabric_table[outcome].eq(metabric_source).all().all(), "a METABRIC outcome differs"
same_columns = list(tcga_table.columns) == list(metabric_table.columns)
assert same_columns, "the two cohort tables have different columns"

## Train and test split

We set aside a share of the TCGA patients (`TEST_FRACTION` in the settings) as the test set. The split is made once, by patient, with a fixed seed, and stratified by the event flag so that both parts have the same share of deaths. The following notebooks use the development set for every choice. The test file is written here and is not opened again until the final model is frozen. The output gives the size of each part and the SHA-256 of its sorted patient list, which identifies the split without showing an identifier.

In [ ]:
development_ids, test_ids = train_test_split(tcga_table.index.to_numpy(), test_size=TEST_FRACTION,
                                             stratify=tcga_table["os_event"],
                                             random_state=SPLIT_SEED)
parts = {"tcga_development": tcga_table.loc[np.sort(development_ids)],
         "tcga_test": tcga_table.loc[np.sort(test_ids)],
         "metabric": metabric_table}
processed_dir = DATA / "processed"
processed_dir.mkdir(exist_ok=True)
split_record = {}
for name, table in parts.items():
    path = processed_dir / f"{name}.parquet"
    table.to_parquet(path)
    id_list = "\n".join(table.index).encode()
    split_record[name] = {"patients": len(table), "deaths": int(table["os_event"].sum()),
                          "deaths (%)": round(100 * table["os_event"].mean(), 1),
                          "patient list sha256": hashlib.sha256(id_list).hexdigest(),
                          "file sha256": file_hash(path, "sha256")}
(RESULTS / "split.json").write_text(json.dumps(split_record, indent=1))
pd.DataFrame(split_record).T

In [ ]:
# The test set can only assess the final model fairly if none of its patients took part in
# development.
assert set(development_ids).isdisjoint(test_ids), "a patient is in both parts"
assert len(development_ids) + len(test_ids) == len(tcga_table), "a patient is in neither part"

## Run record

The last block records what produced the files above: the git commit of this notebook, the Python and package versions, the seed, and the SHA-256 of every input manifest, of the preservation file and of every output file. A second run with the data present reproduces the same hashes.

In [ ]:
packages = {}
for name in ["numpy", "pandas", "pyarrow", "openpyxl", "requests", "scikit-learn",
             "scikit-survival", "matplotlib"]:
    packages[name] = version(name)
output_hashes = {}
for name, record in split_record.items():
    output_hashes[name] = record["file sha256"]
changed = git("status", "--porcelain", "--untracked-files=no").splitlines()
run_record = {"git commit": git("rev-parse", "HEAD"),
              "tracked files changed since that commit": changed,
              "python": sys.version.split()[0],
              "packages": packages,
              "seeds": {"split": SPLIT_SEED},
              "input manifests (sha256)": sources_table["manifest sha256"].to_dict(),
              "preservation file (sha256)": file_hash(preservation_path, "sha256"),
              "output files (sha256)": output_hashes}
(RESULTS / "run_record.json").write_text(json.dumps(run_record, indent=1))
print(json.dumps(run_record, indent=1))